# Do bounded confidence or a co-evolving network change the controversy-axis result?

Two A/B backtests for [Lightningfish](https://github.com/rajul-kk/LightningFish),
run on Kaggle's free T4 GPU — same pattern as `kaggle_controversy.ipynb`
(qwen2.5:7b served locally by Ollama, no API key, $0), sized the same way
(24 agents, 3-4 rounds).

## What's being tested

METHODOLOGY.md's calibrated controversy run found the simulation's crowd-split
prediction lands **below chance** (38% vs. a 53% best baseline, n=74). Two
mechanisms were proposed to make the crowd's split more realistic:

1. **Bounded confidence** (Hegselmann-Krause): T3's herding update gates on
   `confidence_bound`, ignoring targets too far from an agent's own opinion.
   Already tested here (see the "bounded confidence" section below) — **it
   doesn't move the needle**: 48% (off) vs 52% (on), both below the 62%
   baseline, both non-significant (p=0.98, p=0.92), and the 4-point gap is a
   coin-flip's worth of per-event churn, not signal. Logged in
   METHODOLOGY.md's "Worked results" table.
2. **Co-evolving follower network**: agents drop a followed peer once their
   opinion drifts past a bound and refill from closer accounts, so echo
   chambers form dynamically instead of being fixed at round 0
   (`rewire_follower_graph`, opt-in via `coevolving_network=True`). **Not yet
   tested against real events** — this notebook adds that arm.

Both are mechanism tests, not claims that either "works": the honest prior,
given every axis on this domain has failed the ladder so far including
bounded confidence, is that the network arm probably won't move it either.
Reporting that plainly either way is the point.


---
## What the data is

**Source:** the [Hacker News Algolia API](https://hn.algolia.com/api) — free,
unauthenticated, ~10k requests/hour. No key, no scraping.

**Sample:** settled stories at least 24 hours old. `PULL_LIMIT` needs to be
generous — a local run at `limit=60` only cleared 20 scoreable events (33%),
well under the harness's 15/15 minimum split, so this uses 250.

**The seed each agent reads** — strictly submission-time fields, never the
outcome (title, author + karma, url domain, type, self-text). Point-in-time
safety is enforced in code and tested.

**The label:** `num_comments / points` at settlement — ratio >= 0.7 is
"contested," < 0.4 is "consensus," the gap and anything under 20 points is
skipped. See `kaggle_controversy.ipynb`'s data section for the full rationale.


---
## What's different between the arms

All three runs share the exact same event pull, calibration/evaluation split
(deterministic hash of event id), and threshold-derivation logic — only the
flag passed to `_run_hn_controversy_calibrated` differs:

| Run | bounded_confidence | coevolving_network |
|---|---|---|
| `hn-controversy-calibrated` | True (default) | False (default) |
| `hn-controversy-calibrated-nobc` | False | False |
| `hn-controversy-calibrated-network` | True (default) | True |

The network arm's control is the first row, already run in the bounded-
confidence test — the harness's run-cache keys on both flags (`:bc1`, `:net1`
suffixes), so nothing here re-simulates the bc-only arm, only the new
network-enabled one.


---
## 1. Setup

Sidebar: **Accelerator -> GPU** and **Internet -> On**.

`zstd` first: Ollama's installer needs it to extract, Kaggle's image doesn't
ship it, and without this the install fails quietly and surfaces later as a
confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest praw yfinance

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms bounded confidence, the
# CachingAdapter kwarg-forwarding fix, and the calibrated-threshold code are
# actually present in this clone (all pushed in commit 17bcc07).
!python -m pytest tests/core tests/hn -q 2>&1 | tail -5


---
## 2. Configuration


In [ ]:
PULL_LIMIT = 250      # stories to pull; expect roughly 1 in 3 to be scoreable
N_AGENTS   = 24        # matches kaggle_controversy.ipynb's validated GPU size
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}, two arms")


### Throughput check

~26 model calls per event, times two arms. Confirm the per-call cost before
committing — if this shows double digits you're on CPU regardless of what the
assert above said.


In [ ]:
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event  ->  ~{per_call*26*2:.0f}s per event-pair (both arms)")


---
## 3. Run all three arms

Two of these (`hn-controversy-calibrated`, `hn-controversy-calibrated-nobc`)
are the bounded-confidence A/B; the third (`hn-controversy-calibrated-network`)
adds the co-evolving-network arm, compared against the first row as its
control.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_on.log


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-nobc {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_off.log


### Co-evolving network arm

Same events, same split, `bounded_confidence=True` (the default) held
constant — the only thing this changes vs. the first run above is whether the
follower graph rewires each round.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-network {PULL_LIMIT} 2>&1 | tee /kaggle/working/network_on.log


### Reading the logs

- `X of Y events have a controversy direction` — how much the points floor and
  gap zone discarded (should match across all three logs — same pull).
- `split: N calibration / M evaluation` — abort if either is under ~15.
- `calibration stddev range: a-b, median (threshold) = t` — the derived
  cutoff; each arm generally calibrates to a slightly different threshold.
- The final report block in each log: `beats_baselines` and `p_value_vs_best`.

**Bounded confidence** (`bc_on.log` vs `bc_off.log`): already run once, result
was a non-effect (see the intro). Re-running should land close to 48%/52%
again if you want to sanity-check reproducibility, but don't expect a
different verdict — n=42 leaves individual events worth ~2.4 points each.

**Co-evolving network** (`network_on.log` vs `bc_on.log` as its control):

- **Materially higher accuracy, or `beats_baselines` flips to PASS** — the
  network-rewiring mechanism is worth carrying forward as real, pending a
  larger n to confirm it's not the same kind of noise bounded confidence
  turned out to be.
- **About the same, or worse** — consistent with the rest of this project's
  findings on this domain: HN reception is driven by who posts and replies
  early, not by which local social mechanic the crowd runs on. Log it as a
  **fails** in METHODOLOGY.md next to the bounded-confidence rows, with the
  same "diff the two reports event-by-event before trusting the point gap"
  discipline that showed the bounded-confidence gap was churn, not signal.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache


All three logs and the run cache are saved to `/kaggle/working/` — download
them from the notebook's Output tab. The cache holds every simulated run's
final distribution keyed by which flags were on (`:bc1`, `:net1` suffixes),
so a future question about these same events costs nothing to re-score.

Whatever the network arm's result, it belongs in
[METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md)
next to the bounded-confidence rows — a negative is exactly as reportable as
the other rows in that table.
